 Descripción:
En este notebook, vamos a predecir si el ingreso de una persona es superior a 50k o inferior a 50k usando varias características como edad, educación y ocupación.

El conjunto de datos que vamos a usar es el dataset Adult del censo de ingresos de Kaggle que contiene aproximadamente 32561 filas y 15 características 

El dataset contiene las etiquetas que tenemos que predecir y las etiquetas son discretas y binarias. Entonces, el problema que tenemos es de tipo Clasificación Supervisada.


## Paso 0: Cargar librerías y dataset

In [47]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [48]:
# Importar dataset
dataset = pd.read_csv('Datasets Primer Parcial/5-Adult (Census Income)/adult.csv')

## Paso 1: Análisis descriptivo

In [49]:
# Vista previa del dataset
dataset.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [50]:
# Forma del dataset
print('Filas: {} Columnas: {}'.format(dataset.shape[0], dataset.shape[1]))

Filas: 32561 Columnas: 15


In [51]:
# Tipo de datos de las características
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [52]:
# Resumen estadístico
dataset.describe().T

,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
fnlwgt,32561.0,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education.num,32561.0,10.080679,2.572720,1.0,9.0,10.0,12.0,16.0
capital.gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital.loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours.per.week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0


In [53]:
# Verificar valores nulos
round((dataset.isnull().sum() / dataset.shape[0]) * 100, 2).astype(str) + ' %'

age               0.0 %
workclass         0.0 %
fnlwgt            0.0 %
education         0.0 %
education.num     0.0 %
marital.status    0.0 %
occupation        0.0 %
relationship      0.0 %
race              0.0 %
sex               0.0 %
capital.gain      0.0 %
capital.loss      0.0 %
hours.per.week    0.0 %
native.country    0.0 %
income            0.0 %
dtype: str

In [54]:
# Verificar '?' en dataset
round((dataset.isin(['?']).sum() / dataset.shape[0])
      * 100, 2).astype(str) + ' %'

age                0.0 %
workclass         5.64 %
fnlwgt             0.0 %
education          0.0 %
education.num      0.0 %
marital.status     0.0 %
occupation        5.66 %
relationship       0.0 %
race               0.0 %
sex                0.0 %
capital.gain       0.0 %
capital.loss       0.0 %
hours.per.week     0.0 %
native.country    1.79 %
income             0.0 %
dtype: str

In [55]:
# Contar las categorías de etiquetas
income = dataset['income'].value_counts(normalize=True)
round(income * 100, 2).astype('str') + ' %'

income
<=50K    75.92 %
>50K     24.08 %
Name: proportion, dtype: str

Observaciones:


El dataset no tiene valores nulos, pero contiene valores faltantes en forma de '?' que necesitan ser preprocesados.
  


El dataset está desbalanceado, ya que la característica dependiente 'income' contiene 75.92% de valores con ingreso menor a 50k y 24.08% de valores con ingreso mayor a 50k.


##  Preprocesamiento de Datos

### 1: Corregir valores '?' en el dataset

In [56]:
dataset = dataset.replace('?', np.nan)

In [57]:
# Verificar valores nulos
round((dataset.isnull().sum() / dataset.shape[0]) * 100, 2).astype(str) + ' %'

age                0.0 %
workclass         5.64 %
fnlwgt             0.0 %
education          0.0 %
education.num      0.0 %
marital.status     0.0 %
occupation        5.66 %
relationship       0.0 %
race               0.0 %
sex                0.0 %
capital.gain       0.0 %
capital.loss       0.0 %
hours.per.week     0.0 %
native.country    1.79 %
income             0.0 %
dtype: str

In [58]:
columns_with_nan = ['workclass', 'occupation', 'native.country']

In [59]:
for col in columns_with_nan:
    dataset[col].fillna(dataset[col].mode()[0], inplace=True)

### 2: Codificación de Etiquetas

In [60]:
from sklearn.preprocessing import LabelEncoder

In [61]:
from sklearn.preprocessing import LabelEncoder

# Verificar y codificar columnas no numéricas
encoders = {}
columns_to_encode = dataset.select_dtypes(include=['object']).columns

print(f"Columnas a codificar: {list(columns_to_encode)}\n")

for col in columns_to_encode:
    encoder = LabelEncoder()
    print(f"Codificando '{col}'...")
    values = dataset[col].astype(str).fillna('missing')
    dataset[col] = encoder.fit_transform(values)
    encoders[col] = encoder
    print(f"  ✓ Completado: {len(encoder.classes_)} clases únicas\n")

print("✓ Todas las columnas codificadas")
print("\nTipos DESPUÉS:")
print(dataset.dtypes)

Columnas a codificar: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']

Codificando 'workclass'...
  ✓ Completado: 9 clases únicas

Codificando 'education'...
  ✓ Completado: 16 clases únicas

Codificando 'marital.status'...
  ✓ Completado: 7 clases únicas

Codificando 'occupation'...
  ✓ Completado: 15 clases únicas

Codificando 'relationship'...
  ✓ Completado: 6 clases únicas

Codificando 'race'...
  ✓ Completado: 5 clases únicas

Codificando 'sex'...
  ✓ Completado: 2 clases únicas

Codificando 'native.country'...
  ✓ Completado: 42 clases únicas

Codificando 'income'...
  ✓ Completado: 2 clases únicas

✓ Todas las columnas codificadas

Tipos DESPUÉS:
age               int64
workclass         int64
fnlwgt            int64
education         int64
education.num     int64
marital.status    int64
occupation        int64
relationship      int64
race              int64
sex               int64
capital.gain      int64
cap

### 3: Selección de Características

In [62]:
X = dataset.drop('income', axis=1)
Y = dataset['income']

# Verificar tipos de datos en X
print("Tipos de datos en X:")
print(X.dtypes)

Tipos de datos en X:
age               int64
workclass         int64
fnlwgt            int64
education         int64
education.num     int64
marital.status    int64
occupation        int64
relationship      int64
race              int64
sex               int64
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country    int64
dtype: object


In [63]:
from sklearn.ensemble import ExtraTreesClassifier
selector = ExtraTreesClassifier(random_state=42)

In [64]:
# Verificar que X es numérico antes de entrenar
print("Verificando tipos en X:")
print(X.dtypes)
print("\n¿Hay columnas no numéricas?:")
print(X.select_dtypes(include=['object']).columns.tolist())

# Convertir cualquier columna restante a numérico
X = X.astype('float64', errors='ignore')

print("\nEntrenando selector...")
selector.fit(X, Y)
print("✓ Selector entrenado exitosamente")

Verificando tipos en X:
age               int64
workclass         int64
fnlwgt            int64
education         int64
education.num     int64
marital.status    int64
occupation        int64
relationship      int64
race              int64
sex               int64
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country    int64
dtype: object

¿Hay columnas no numéricas?:
[]

Entrenando selector...
✓ Selector entrenado exitosamente


In [65]:
feature_imp = selector.feature_importances_

In [66]:
for index, val in enumerate(feature_imp):
    print(index, round((val * 100), 2))

0 15.42
1 4.24
2 16.46
3 3.66
4 8.92
5 7.29
6 7.44
7 9.22
8 1.45
9 2.95
10 8.74
11 2.75
12 9.51
13 1.96


In [67]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             32561 non-null  float64
 1   workclass       32561 non-null  float64
 2   fnlwgt          32561 non-null  float64
 3   education       32561 non-null  float64
 4   education.num   32561 non-null  float64
 5   marital.status  32561 non-null  float64
 6   occupation      32561 non-null  float64
 7   relationship    32561 non-null  float64
 8   race            32561 non-null  float64
 9   sex             32561 non-null  float64
 10  capital.gain    32561 non-null  float64
 11  capital.loss    32561 non-null  float64
 12  hours.per.week  32561 non-null  float64
 13  native.country  32561 non-null  float64
dtypes: float64(14)
memory usage: 3.5 MB


In [68]:
X = X.drop(['workclass', 'education', 'race', 'sex',
            'capital.loss', 'native.country'], axis=1)

### 4: Escalado de Características

In [69]:
from sklearn.preprocessing import StandardScaler

In [70]:
for col in X.columns:
    scaler = StandardScaler()
    X[col] = scaler.fit_transform(X[col].values.reshape(-1, 1))

### 5: Corregir dataset desbalanceado usando Oversampling

In [71]:
round(Y.value_counts(normalize=True) * 100, 2).astype('str') + ' %'

income
0    75.92 %
1    24.08 %
Name: proportion, dtype: str

In [72]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

In [73]:
ros.fit(X, Y)

,sampling_strategy,'auto'
,random_state,42
,shrinkage,None


In [74]:
X_resampled, Y_resampled = ros.fit_resample(X, Y)

In [75]:
round(Y_resampled.value_counts(normalize=True) * 100, 2).astype('str') + ' %'

income
0    50.0 %
1    50.0 %
Name: proportion, dtype: str

### 6: Crear división entrenamiento-prueba

In [76]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X_resampled, Y_resampled, test_size=0.2, random_state=42)

In [77]:
print("Forma de X_train:", X_train.shape)
print("Forma de X_test:", X_test.shape)
print("Forma de Y_train:", Y_train.shape)
print("Forma de Y_test:", Y_test.shape)

Forma de X_train: (39552, 8)
Forma de X_test: (9888, 8)
Forma de Y_train: (39552,)
Forma de Y_test: (9888,)
